## YOLO

- Python을 기반하는 영상인식 라이브러리
- 라즈베리파이 사용 가능
- 라즈베리파이 5기준
    - YOLOv5 : 10~20FPS
    - YOLOv8n : 8~15FPS
    - YOLOv8s : 3~7FPS
- YOLO 사용시 대부분 Jeston Nano라는 NVIDIA 제품 사용

### 객체 탐지
- Object Detection
    - 이미지, 영상(한 프레임 이미지) 위치와 클래스 모두 예측
    - x,y,w,h 영역안에 물체를 감지
    - 자율주행, 로봇, CCTV 분석 등 활용

- 원리
    - 물체후보영역 추출
    - 각 영역 분류
    - 위 원리를 더 세분화, 그리드화 시킨 알고리즘 활용 : YOLO

### YOLO 설치
- Numpy, opencv-python, Pytorch, Pytorch Vision 라이브러리 필수
- 미리 설치 권고
- PyTorch GPU버전 미리 설치

### 1. 라이브러리 로드

In [1]:
from ultralytics import YOLO

- 최초 로드시 C:\Users\User\appdata\

### 2. 사전학습 모델 다운로드

In [2]:
# n(Namo), S(Small), m(MEdium), l(Large), x(eXtraLarge)\
model = YOLO('yolo11n.pt')

### 3. 이미지 예측

In [3]:
result = model('./bus.jpg')


image 1/1 c:\SourceBank\iot-python-2026\day11\bus.jpg: 640x480 4 persons, 1 bus, 36.0ms
Speed: 1.8ms preprocess, 36.0ms inference, 14.7ms postprocess per image at shape (1, 3, 640, 480)


### 4. 결과확인

- yolo11n은 총 80 물체 인식

In [4]:
result[0].show()

### 5. 결과 이미지 저장

In [5]:
result[0].save(filename='1.jpg')

'1.jpg'

### 6. 데이터폴더 전체 인식

In [18]:
results = model('./data/')

for i, result in enumerate(results):
    result.save(filename=f'result_{i}.jpg')


image 1/2 c:\SourceBank\iot-python-2026\day11\data\bus.jpg: 640x480 4 persons, 1 bus, 8.6ms
image 2/2 c:\SourceBank\iot-python-2026\day11\data\image.png: 384x640 2 persons, 1 tie, 37.9ms
Speed: 1.6ms preprocess, 23.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)


### 7. 동영상 인식

In [ ]:
results = model.predict(
    source='./sample01.mp4',
    show=True,       # 화면에 실시간으로 재생하며 보여줍니다.
    save=True,       # 'runs/detect/predict/' 폴더에 결과 영상이 저장됩니다.
    stream=True      # 영상이 길 경우 메모리 부족을 방지하기 위해 스트리밍 처리합니다.
)

# 2. 실시간 재생 및 중지 제어를 위한 루프
for r in results:
    # 내부적으로 윈도우 창이 켜져 있을 때 'q' 키를 누르면 종료할 수 있도록 설정
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 3. 모든 창 닫기
cv2.destroyAllWindows()

### 8. 동영상 플레이 하면서 물체 인식

In [27]:
# 영상출력 기본
import cv2
import time
from ultralytics import YOLO
model = YOLO('yolo11n.pt')

#cap = cv2.VideoCapture('./sample/sample02.mp4')
cap = cv2.VideoCapture(0)
prev_time = 0

while True:
    # 프레임 읽기
    ret, frame = cap.read()
    if not ret: break

    # YOLO 영상추출
    results = model(frame, conf = 0.5)
    pred_frame = results[0].plot()

    curr_time = time.time()
    fps = 1 / (curr_time - prev_time)
    prev_time = curr_time

    # 2. FPS 출력
    cv2.putText(
        pred_frame,
        f'FPS : {fps:.1f}',
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.imshow('YOLO Player', pred_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()


0: 480x640 (no detections), 35.7ms
Speed: 1.2ms preprocess, 35.7ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.0ms
Speed: 1.2ms preprocess, 9.0ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 10.8ms
Speed: 0.9ms preprocess, 10.8ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 7.5ms
Speed: 0.9ms preprocess, 7.5ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 8.2ms
Speed: 2.6ms preprocess, 8.2ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.8ms
Speed: 1.1ms preprocess, 9.8ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 8.3ms
Speed: 0.8ms preprocess, 8.3ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 8.3ms
Speed: 0.9ms preprocess, 8.3ms inference, 0.

### 9. 침입탐지

In [34]:
# 영상출력 기본
import cv2
import time
from ultralytics import YOLO
import winsound

model = YOLO('yolo11n.pt')
cap = cv2.VideoCapture(0)

while True:
    # 프레임 읽기
    ret, frame = cap.read()
    if not ret: break

    # YOLO 영상추출
    results = model(frame, conf = 0.5)
    detected = False

    # 사람 인식
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]

        if label == 'person':
                detected = True
                # 사람을 한 명이라도 찾았다면 더 돌 필요가 없으므로 break
                break
    
    pred_frame = results[0].plot()

    if detected:
        cv2.putText(
            pred_frame,
            'Person Detected',
            (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255), # GBR
            2
        )
        winsound.Beep(1000, 300)    # 경고음

    cv2.imshow('Detection Alarm', pred_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()


0: 480x640 1 person, 16.9ms
Speed: 1.3ms preprocess, 16.9ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 9.7ms
Speed: 1.1ms preprocess, 9.7ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 8.6ms
Speed: 0.9ms preprocess, 8.6ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 8.7ms
Speed: 1.1ms preprocess, 8.7ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 9.4ms
Speed: 0.9ms preprocess, 9.4ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 7.7ms
Speed: 0.9ms preprocess, 7.7ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 9.4ms
Speed: 1.0ms preprocess, 9.4ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 9.8ms
Speed: 0.9ms preprocess, 9.8ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)


### 1. COCO 객체

-  Common Object in COntext : 컴퓨터 비전에서 가장 유명한 데이터셋 중 하나

- YOLO 기본모델은 COCO DataSet만인식

- 80가지 이외의 물체를 인식하려면
    - 직접 학습시킨 모델 생성
    - 타 사이트에서 제공하는 모델 사용 : https://huggingface.co/

- 이미지, 영상 인식

- YOLO 설치

    ```powershell
    > pip install ultralytics huggingface_hub
    ```

### 11. 화재 감지

In [35]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id = 'keremberke/yolov8n-fire-detection',
    filename='best.pt'
)

c:\SourceBank\iot-python-2026\iot-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6a0fcdc1-34b1825e5f97f592271d6bf0;b9213458-9278-45b6-bf6d-8cf55069c8f9)

Repository Not Found for url: https://huggingface.co/keremberke/yolov8n-fire-detection/resolve/main/best.pt.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [ ]:
import cv2
import time
from ultralytics import YOLO
import winsound

model = YOLO('firedetect-11s.pt')
cap = cv2.VideoCapture('./sample/sample03.mp4')

while True:
    ret, frame = cap.read()
    if not ret: break

    results = model(frame)
    pred_frame = results[0].plot()
    
    cv2.imshow('Fire Detection', pred_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()


0: 384x640 1 Smoke, 9.4ms
Speed: 0.9ms preprocess, 9.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Fire, 9.0ms
Speed: 1.0ms preprocess, 9.0ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Fire, 9.1ms
Speed: 1.4ms preprocess, 9.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Fire, 8.4ms
Speed: 0.9ms preprocess, 8.4ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Fire, 11.1ms
Speed: 1.1ms preprocess, 11.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Fire, 12.1ms
Speed: 1.1ms preprocess, 12.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Fire, 14.7ms
Speed: 0.9ms preprocess, 14.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Fire, 7.5ms
Speed: 0.9ms preprocess, 7.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Fir